# DiffusionBlocks, Explained Simply

Modern networks are trained **end-to-end**: to update any layer, the activations of *every* layer
are held in memory for backpropagation. So training memory grows with depth — a real bottleneck for
large models.

[**DiffusionBlocks**](https://arxiv.org/abs/2506.14202) (Shing, Koyama & Akiba, ICLR 2026) asks a
simple question: what if each *block* of the network could be trained **on its own**? Then we only
ever hold one block's activations — roughly **1/B** the memory for B blocks.

The trick is to treat a residual network as a **diffusion model** walking from noise to the answer.
Each block owns one stretch of that walk and can be trained independently of the others.

This notebook is a tiny, laptop-friendly demonstration on MNIST. We'll show that block-wise training
**nearly matches** a standard classifier's accuracy while using a **fraction of the training
memory** — and, just as importantly, build intuition for *why* it works.

> This is a teaching toy, deliberately small. It is not the paper's ViT/CIFAR-100, DiT, or
> text-generation scale. See the caveats at the end.

In [ ]:
import random

import numpy as np
import torch
import matplotlib.pyplot as plt

import mini_db

random.seed(0)
torch.manual_seed(0)
np.random.seed(0)

# A slightly wider backbone (H=256) so both models have enough capacity on MNIST.
cfg = mini_db.Config(H=256)
device = mini_db.get_device()
EPOCHS = 14  # compute budget; see note below on why this is a *fair* comparison

print("device:", device)
print(f"backbone: {cfg.num_layers} residual layers split into {cfg.num_blocks} blocks "
      f"({cfg.num_layers // cfg.num_blocks} layers/block), width H={cfg.H}")

train_loader, test_loader = mini_db.load_mnist(n_train=8000, n_test=2000, batch_size=128)
print(f"MNIST: {len(train_loader.dataset)} train / {len(test_loader.dataset)} test images")

## 1. The baseline you already know

A standard residual-MLP classifier: image → logits, trained with cross-entropy and backprop through
**all** layers at once. This is our familiar, memory-hungry reference point.

It uses the **exact same backbone** (same layers, width, and block count) that DiffusionBlocks will
use — so later, the only things that differ are the *training objective* and the *training loop*,
not the architecture.

In [ ]:
torch.manual_seed(0)
baseline = mini_db.PlainClassifier(cfg)
base_hist = mini_db.train_baseline(baseline, train_loader, epochs=EPOCHS, lr=1e-3, device=device)
base_acc = mini_db.evaluate(baseline, test_loader, device=device, diffusion=False)
print(f"baseline test accuracy: {base_acc:.3f}")

## 2. The reframe: classification as denoising

Here's the shift in perspective. Give each digit class a target vector `z₀` (a learned embedding).
Now corrupt it with noise:

$$z_\sigma = z_0 + \sigma \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

The model, **conditioned on the image**, learns to remove the noise. Whichever clean class-vector it
lands on *is* the predicted digit. The noise level `σ` plays the role of "how far from the answer
are we":

- **large σ** → almost pure noise (the *early* part of the walk),
- **small σ** → nearly the clean answer (the *late* part of the walk).

We split the σ-axis into `B` ranges and put **one block in charge of each range**. Because a block is
always fed the *analytic* noisy vector `z_σ` for its own range — never the previous block's output —
it can be trained **completely independently**. At inference we chain the blocks together, walking
σ from high to low, one Euler step per block.

(We keep the EDM preconditioning from Karras et al. 2022 so each block has a well-scaled target at
every σ; it lives in `mini_db.edm_scalings` if you want to peek.)

### A fair comparison

In our block-wise loop, **one training step touches only one block** (2 layers) instead of the full
network (6 layers). To keep total compute equal, DiffusionBlocks runs `B`× more steps. So training
both models for the same `EPOCHS` value means roughly **equal FLOPs** — the comparison isolates the
*training method*, not the compute budget. The payoff is in **memory**, which we measure below.

In [ ]:
torch.manual_seed(0)
db_model = mini_db.DiffusionClassifier(cfg)
db_hist = mini_db.train_diffusionblocks(db_model, train_loader, epochs=EPOCHS, lr=1e-3, device=device, seed=0)
db_acc = mini_db.evaluate(db_model, test_loader, device=device, diffusion=True)
print(f"DiffusionBlocks test accuracy: {db_acc:.3f}")

## 3. Watch the denoising walk

Let's make the idea concrete. We take one test image, start from pure noise, and let the blocks walk
the latent vector `z` down the σ-ladder. Classification only depends on the *direction* of `z`
(the logits are dot-products with the unit-norm class targets), so we plot unit-normalized
directions and fit a 2D PCA on the 10 class targets. Watch the red path start at noise and land on
the true class point (highlighted in green).

In [ ]:
# cfg.num_blocks

In [ ]:
db_model.eval()
with torch.no_grad():
    embed = db_model.class_embeddings().cpu().numpy()
    # Take artbitrary index 2 from test_loader
    # images, labels = next(iter(test_loader))
    images, labels = [(img, label) for img, label in test_loader][2]
    img = images[:1].to(device)
    true = labels[0].item()
    steps = cfg.num_blocks + 1
    sigmas = mini_db.get_inference_sigmas(steps, cfg).to(device)
    z = torch.randn(1, cfg.H, device=device) * (1 + sigmas[0] ** 2) ** 0.5
    traj = [z.cpu().numpy()[0]]
    for i in range(steps - 1):
        sig = sigmas[i].expand(1)
        logits = db_model.denoise(img, z, sig)
        denoised = torch.softmax(logits, -1) @ db_model.class_embeddings()
        z = z + (sigmas[i + 1] - sigmas[i]) * (z - denoised) / sig[:, None] # eq. 5
        traj.append(z.cpu().numpy()[0])
    pred = db_model.denoise(img, z, sigmas[-1].expand(1)).argmax(-1).item()

traj = np.array(traj)

# z starts enormous (sigma ~ tens), which would dwarf the unit-norm class targets and
# squash everything into one corner. Classification only depends on *direction*
# (logits = denoised . unit-class-embeddings), so we view unit-normalized directions and
# fit the 2D PCA on the class targets, so the 10 classes spread out.
def unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)

emb_u, traj_u = unit(embed), unit(traj)
mean = emb_u.mean(0)
_, _, Vt = np.linalg.svd(emb_u - mean, full_matrices=False)
P = Vt[:2].T
emb2 = (emb_u - mean) @ P
tr2 = (traj_u - mean) @ P

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(emb2[:, 0], emb2[:, 1], s=520, c="whitesmoke", edgecolors="silver", zorder=1)
for k in range(10):
    color = "seagreen" if k == true else "dimgray"
    weight = "bold" if k == true else "normal"
    ax.text(emb2[k, 0], emb2[k, 1], str(k), ha="center", va="center",
            fontsize=12, color=color, fontweight=weight, zorder=10)
ax.plot(tr2[:, 0], tr2[:, 1], "-", color="crimson", lw=1.8, zorder=3)
ax.scatter(tr2[1:-1, 0], tr2[1:-1, 1], s=30, color="crimson", zorder=4)
ax.scatter(tr2[0, 0], tr2[0, 1], s=90, color="black", zorder=5, label="start (noise)")
ax.scatter(tr2[-1, 0], tr2[-1, 1], s=320, marker="*", facecolors="none",
           edgecolors="crimson", linewidths=1.8, zorder=6,
           label=f"end (predicted {pred})")
ax.legend(loc="best", framealpha=0.9)
ax.set_title(f"denoising walk in label-embedding space  |  true = {true}, predicted = {pred}")
ax.set_xlabel("class-target PC 1"); ax.set_ylabel("class-target PC 2")
plt.show()

## 4. Results: accuracy and memory

First accuracy. The DiffusionBlocks loss curve is logged once per pass and there are `B`× more
passes, so we plot both against *training progress* (0 → 1) to compare fairly.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

base_x = np.linspace(0, 1, len(base_hist["loss"]))
db_x = np.linspace(0, 1, len(db_hist["loss"]))
ax[0].plot(base_x, base_hist["loss"], label="baseline (end-to-end)")
ax[0].plot(db_x, db_hist["loss"], label="DiffusionBlocks (block-wise)")
ax[0].set_title("training loss"); ax[0].set_xlabel("training progress"); ax[0].set_ylabel("loss")
ax[0].legend()

bars = ax[1].bar(["baseline", "DiffusionBlocks"], [base_acc, db_acc], color=["gray", "crimson"])
ax[1].set_ylim(0, 1); ax[1].set_title("test accuracy")
for b, v in zip(bars, [base_acc, db_acc]):
    ax[1].text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center")
plt.show()

print(f"baseline = {base_acc:.3f}   DiffusionBlocks = {db_acc:.3f}   gap = {base_acc - db_acc:+.3f}")

Now the headline: **memory**. Each training step of DiffusionBlocks keeps only **one block** in the
autograd graph, while end-to-end keeps **all B**. We measure the activation bytes held for backprop.

In [ ]:
images, _ = next(iter(train_loader))
images = images.to(device)
z = torch.randn(images.shape[0], cfg.H, device=device)
sigma = torch.full((images.shape[0],), 1.0, device=device)

full_bytes = mini_db.activation_bytes(db_model, images, z, sigma, blocks_in_graph=cfg.num_blocks)
one_bytes = mini_db.activation_bytes(db_model, images, z, sigma, blocks_in_graph=1)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["end-to-end\n(all blocks)", "DiffusionBlocks\n(one block)"],
              [full_bytes / 1e6, one_bytes / 1e6], color=["gray", "crimson"])
ax.set_ylabel("activation memory per step (MB)")
ax.set_title(f"~{full_bytes / one_bytes:.1f}x less activation memory")
for b, v in zip(bars, [full_bytes / 1e6, one_bytes / 1e6]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")
plt.show()

print(f"analytic activation memory per step:")
print(f"  end-to-end (all {cfg.num_blocks} blocks) = {full_bytes/1e6:.2f} MB")
print(f"  DiffusionBlocks (one block)        = {one_bytes/1e6:.2f} MB")
print(f"  ratio = {full_bytes/one_bytes:.1f}x  (equals the number of blocks B = {cfg.num_blocks})")

## Takeaways

- DiffusionBlocks lands in the **same ballpark** as the end-to-end baseline (within roughly ten
  points on this tiny toy) while only ever holding **one block's activations** in the autograd
  graph → about **1/B** the training memory. The block-wise model is solving a strictly harder
  problem (denoising), so a small gap at this scale is expected; the paper closes it at real scale.
- Each block is trained **independently**: it is fed the analytic noisy embedding for its own
  σ-range and never sees the other blocks' outputs during training. We verified this in the test
  suite (`test_diffusion_denoise_block_independent_of_other_blocks`).
- At inference the blocks chain together into a short denoising walk from noise to the answer.

### Honest caveats

- The two models share the **same backbone** but use **different objectives** (plain cross-entropy
  vs diffusion-denoising). The comparison isolates the *training method* (all-at-once vs
  one-block-at-a-time), not a perfectly identical loss.
- This is a deliberately tiny MNIST toy. The small accuracy gap here is a property of the toy scale;
  the paper reports parity (and sometimes gains) at real scale across ViT/DiT/text models.
- "Memory" here is a clean **activation-accounting proxy** — the quantity that actually scales with
  depth. Real peak-memory measurement is noisier and device-specific; the proxy is what makes the
  1/B story exact and reproducible.
- The headline benefit is **memory during training**, not wall-clock speed: block-wise training does
  the same total work, just one block at a time.

## (Stretch) Varying the number of blocks B

More blocks → less memory per step, but each block owns a narrower slice of the walk. Let's sweep
`B` over the divisors of the 6-layer trunk (2, 3, 6) and plot the accuracy vs per-step memory
trade-off. This cell trains several models, so it takes a bit longer.

A subtlety: in this demo the number of inference Euler steps equals the number of blocks, so more
blocks also means *finer* integration of the denoising walk. That's why accuracy here tends to
**hold or even improve** as B grows — the trade-off you're really buying is less memory per step in
exchange for more (cheaper) inference steps, not lower accuracy.

(We skip `B=1`: with a single block the inference walk reduces to one Euler step over the whole
σ-range, which is too coarse to integrate well. `B ≥ 2` already tells the trade-off story clearly.)

In [ ]:
results = []
for B in [2, 3, 6]:
    torch.manual_seed(0)
    c = mini_db.Config(H=cfg.H, num_blocks=B)
    m = mini_db.DiffusionClassifier(c)
    mini_db.train_diffusionblocks(m, train_loader, epochs=EPOCHS, lr=1e-3, device=device, seed=0)
    acc = mini_db.evaluate(m, test_loader, device=device, diffusion=True)
    img, _ = next(iter(train_loader)); img = img.to(device)
    zz = torch.randn(img.shape[0], c.H, device=device)
    ss = torch.full((img.shape[0],), 1.0, device=device)
    per_step = mini_db.activation_bytes(m, img, zz, ss, blocks_in_graph=1)
    results.append((B, acc, per_step / 1e6))
    print(f"B={B}: test acc={acc:.3f}, per-step activation memory={per_step/1e6:.2f} MB")

Bs, accs, mems = zip(*results)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Bs, accs, "-o"); ax[0].set_xlabel("number of blocks B")
ax[0].set_ylabel("test accuracy"); ax[0].set_title("accuracy vs B")
ax[1].plot(Bs, mems, "-o", color="crimson"); ax[1].set_xlabel("number of blocks B")
ax[1].set_ylabel("per-step activation memory (MB)"); ax[1].set_title("memory vs B")
plt.show()